In [10]:

"""
============================================================
Customer Complaint Intelligence
Python | Power BI
============================================================
End-to-end pipeline:
  1. Load & inspect raw data
  2. Clean & engineer features
  3. Analyze: top issue types, resolution-time gaps, repeat
     complainants / churn risk
  4. Recommend a priority resolution workflow + estimate impact
  5. Export: cleaned dataset for Power BI, summary tables, charts
 
Run top to bottom in Jupyter, or as a script:
    python complaint_intelligence.py
 
Author: <your name>
Dataset: Kaggle "Customer Support Ticket Dataset" (8,469 tickets)
============================================================
"""
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
 
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
 
RAW_PATH = "customer_support_tickets.csv"
OUT_DIR = "outputs"
 
# ============================================================
# STEP 1 — LOAD & INITIAL INSPECTION
# ============================================================
 
df = pd.read_csv("customer_support_tickets.csv")
 
print("=" * 60)
print("STEP 1: RAW DATA OVERVIEW")
print("=" * 60)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print("\nNulls per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nTicket Status breakdown:")
print(df["Ticket Status"].value_counts())
 
# ------------------------------------------------------------
# KEY DATA QUALITY FINDING (worth stating in your write-up):
# Resolution/Satisfaction/Time-to-Resolution are NULL for any
# ticket that isn't 'Closed' (Open / Pending Customer Response).
# This is NOT missing data to impute — it's structurally absent
# because those tickets haven't been resolved yet. We keep both
# the full ticket-level table (for volume/category analysis) and
# a closed-only subset (for resolution-time analysis).
# ------------------------------------------------------------
 
# ============================================================
# STEP 2 — CLEANING & FEATURE ENGINEERING
# ============================================================
 
print("\n" + "=" * 60)
print("STEP 2: CLEANING")
print("=" * 60)
 
clean = df.copy()
 
# 2.1 Standardize text columns (trim whitespace, consistent case)
text_cols = ["Customer Name", "Customer Email", "Ticket Type",
             "Ticket Subject", "Ticket Status", "Ticket Priority",
             "Ticket Channel"]
for col in text_cols:
    clean[col] = clean[col].astype(str).str.strip()
 
clean["Customer Email"] = clean["Customer Email"].str.lower()
 
# 2.2 Parse dates
clean["Date of Purchase"] = pd.to_datetime(clean["Date of Purchase"], errors="coerce")
clean["First Response Time"] = pd.to_datetime(clean["First Response Time"], errors="coerce")
clean["Time to Resolution"] = pd.to_datetime(clean["Time to Resolution"], errors="coerce")
 
# 2.3 Drop exact duplicate rows (by Ticket ID)
before = len(clean)
clean = clean.drop_duplicates(subset="Ticket ID")
print(f"Removed {before - len(clean)} duplicate Ticket IDs")
 
# 2.4 Engineer: real resolution duration (in hours)
# Time to Resolution and First Response Time are both timestamps,
# so duration = resolution_ts - first_response_ts.
clean["Resolution Hours"] = (
    (clean["Time to Resolution"] - clean["First Response Time"])
    .dt.total_seconds() / 3600
)
 
# 2.5 Data-quality flag: some rows have a resolution timestamp
# earlier than the first-response timestamp (impossible duration).
# This is a known quirk of this dataset (synthetic timestamps were
# generated independently rather than sequentially). We flag and
# exclude these from duration *averages*, but keep them in the
# dataset with a flag column so Power BI can show the data-quality
# rate transparently rather than hiding it.
clean["Valid Duration"] = clean["Resolution Hours"] > 0
 
n_closed = (clean["Ticket Status"] == "Closed").sum()
n_invalid = ((clean["Ticket Status"] == "Closed") & (~clean["Valid Duration"])).sum()
print(f"Closed tickets: {n_closed:,}")
print(f"Closed tickets with invalid (<=0 hr) duration: {n_invalid:,} "
      f"({n_invalid/n_closed:.1%}) -- flagged, excluded from duration averages")
 
# 2.6 Repeat complainant flag (churn risk signal)
email_counts = clean["Customer Email"].value_counts()
clean["Ticket Count for Customer"] = clean["Customer Email"].map(email_counts)
clean["Repeat Complainant"] = clean["Ticket Count for Customer"] > 1
 
n_unique_customers = clean["Customer Email"].nunique()
n_repeat_customers = (email_counts > 1).sum()
pct_repeat_customers = n_repeat_customers / n_unique_customers
print(f"\nUnique customers: {n_unique_customers:,}")
print(f"Repeat complainants (>1 ticket): {n_repeat_customers:,} "
      f"({pct_repeat_customers:.1%} of unique customers)")
 
# 2.7 Age bands (useful Power BI slicer)
clean["Age Band"] = pd.cut(
    clean["Customer Age"],
    bins=[0, 24, 34, 44, 54, 64, 120],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
)
 
# 2.8 Fill non-applicable categorical nulls for BI usability
clean["Resolution"] = clean["Resolution"].fillna("Not yet resolved")
 
print(f"\nFinal cleaned shape: {clean.shape}")
 
# ============================================================
# STEP 3 — ANALYSIS
# ============================================================
 
print("\n" + "=" * 60)
print("STEP 3: ANALYSIS")
print("=" * 60)
 
# 3.1 Top issue types by volume
type_volume = (
    clean["Ticket Type"].value_counts()
    .rename_axis("Ticket Type")
    .reset_index(name="Ticket Count")
)
type_volume["% of Total"] = (type_volume["Ticket Count"] / len(clean) * 100).round(1)
 
top3 = type_volume.head(3)
top3_share = top3["Ticket Count"].sum() / len(clean) * 100
 
print("\nTicket volume by type:")
print(type_volume.to_string(index=False))
print(f"\n>>> Top 3 issue types ({', '.join(top3['Ticket Type'])}) "
      f"drive {top3_share:.1f}% of total complaint volume.")
 
# 3.2 Resolution time by issue type (valid durations only)
valid_closed = clean[(clean["Ticket Status"] == "Closed") & (clean["Valid Duration"])]
 
res_time_by_type = (
    valid_closed.groupby("Ticket Type")["Resolution Hours"]
    .agg(Avg_Hours="mean", Median_Hours="median", N="count")
    .round(2)
    .sort_values("Avg_Hours", ascending=False)
    .reset_index()
)
print("\nAvg resolution time by ticket type (valid durations only):")
print(res_time_by_type.to_string(index=False))
 
slowest = res_time_by_type.iloc[0]
fastest = res_time_by_type.iloc[-1]
gap_ratio = slowest["Avg_Hours"] / fastest["Avg_Hours"]
print(f"\n>>> '{slowest['Ticket Type']}' is the slowest category to resolve "
      f"({slowest['Avg_Hours']:.1f} avg hrs), {gap_ratio:.1f}x the fastest "
      f"category ('{fastest['Ticket Type']}', {fastest['Avg_Hours']:.1f} avg hrs).")
 
# Resolution time by priority (independent useful cut for BI)
res_time_by_priority = (
    valid_closed.groupby("Ticket Priority")["Resolution Hours"]
    .agg(Avg_Hours="mean", N="count")
    .round(2)
    .reindex(["Critical", "High", "Medium", "Low"])
    .reset_index()
)
print("\nAvg resolution time by priority:")
print(res_time_by_priority.to_string(index=False))
 
# 3.3 Repeat complainants as churn risk
repeat_df = clean[clean["Repeat Complainant"]]
repeat_type_mix = (
    repeat_df["Ticket Type"].value_counts(normalize=True) * 100
).round(1)
print(f"\nRepeat complainants make up {pct_repeat_customers:.1%} of unique customers "
      f"but account for {len(repeat_df)} tickets "
      f"({len(repeat_df)/len(clean):.1%} of all ticket volume).")
print("\nTicket type mix among repeat complainants:")
print(repeat_type_mix.to_string())
 
avg_satisfaction_repeat = clean.loc[
    clean["Repeat Complainant"] & clean["Customer Satisfaction Rating"].notna(),
    "Customer Satisfaction Rating"
].mean()
avg_satisfaction_oneoff = clean.loc[
    ~clean["Repeat Complainant"] & clean["Customer Satisfaction Rating"].notna(),
    "Customer Satisfaction Rating"
].mean()
print(f"\nAvg satisfaction (repeat complainants): {avg_satisfaction_repeat:.2f}")
print(f"Avg satisfaction (one-off complainants): {avg_satisfaction_oneoff:.2f}")
 
# ============================================================
# STEP 4 — PRIORITY RESOLUTION WORKFLOW RECOMMENDATION
# ============================================================
 
print("\n" + "=" * 60)
print("STEP 4: RECOMMENDATION & PROJECTED IMPACT")
print("=" * 60)
 
# Logic: flag any OPEN/PENDING ticket from a repeat complainant,
# or from the slowest-resolving ticket type, as "Priority Queue".
clean["Priority Queue Flag"] = (
    (clean["Ticket Status"] != "Closed") &
    (clean["Repeat Complainant"] | (clean["Ticket Type"] == slowest["Ticket Type"]))
)
n_priority_flagged = clean["Priority Queue Flag"].sum()
print(f"Tickets flagged for the new priority queue (open + repeat-customer "
      f"OR slowest-category): {n_priority_flagged:,} "
      f"({n_priority_flagged/len(clean):.1%} of all tickets)")
 
print("""
RECOMMENDATION:
Route tickets to a "Priority Resolution Queue" when EITHER is true:
  (a) the customer is a repeat complainant (>1 ticket on file), or
  (b) the ticket type is the historically slowest category
      identified above.
These tickets get first-response SLA tightened and are auto-escalated
to a senior agent rather than entering the standard FIFO queue.
 
PROJECTED IMPACT (illustrative, to validate against pilot data):
Repeat complainants are a small share of customers but a
disproportionate share of ticket volume and run lower satisfaction.
Resolving their issues faster at first contact is the primary lever
for reducing repeat-contact rate. A modeled 25% reduction in repeat
ticket volume, if first-contact resolution improves enough to stop
one in four repeat contacts, is a reasonable pilot target -- this
script outputs the baseline metrics needed to track that KPI before
and after rollout.
""")
 
baseline_repeat_tickets = len(repeat_df)
projected_reduction = 0.25
projected_tickets_saved = int(baseline_repeat_tickets * projected_reduction)
print(f"Baseline repeat-complainant ticket volume: {baseline_repeat_tickets:,}")
print(f"Projected volume saved at 25% reduction target: {projected_tickets_saved:,} tickets")
 
# ============================================================
# STEP 5 — EXPORTS (Power BI dataset + summary tables + charts)
# ============================================================
 
print("\n" + "=" * 60)
print("STEP 5: EXPORTING OUTPUTS")
print("=" * 60)
 
# 5.1 Full cleaned dataset -> Power BI data source
clean.to_csv(f"{OUT_DIR}/cleaned_tickets_for_powerbi.csv", index=False)
 
# 5.2 Summary tables (also useful as separate Power BI tables / DAX validation)
type_volume.to_csv(f"{OUT_DIR}/summary_ticket_type_volume.csv", index=False)
res_time_by_type.to_csv(f"{OUT_DIR}/summary_resolution_time_by_type.csv", index=False)
res_time_by_priority.to_csv(f"{OUT_DIR}/summary_resolution_time_by_priority.csv", index=False)
 
repeat_summary = pd.DataFrame({
    "Metric": [
        "Unique customers",
        "Repeat complainants (customers)",
        "Repeat complainant rate",
        "Tickets from repeat complainants",
        "Avg satisfaction - repeat complainants",
        "Avg satisfaction - one-off complainants",
        "Tickets flagged for priority queue",
        "Projected repeat-ticket reduction target",
        "Projected tickets saved at target",
    ],
    "Value": [
        n_unique_customers,
        n_repeat_customers,
        f"{pct_repeat_customers:.1%}",
        len(repeat_df),
        round(avg_satisfaction_repeat, 2),
        round(avg_satisfaction_oneoff, 2),
        n_priority_flagged,
        "25%",
        projected_tickets_saved,
    ]
})
repeat_summary.to_csv(f"{OUT_DIR}/summary_churn_risk_workflow.csv", index=False)
 
print("Exported cleaned dataset + 4 summary tables to /outputs")
 
# 5.3 Charts
plt.style.use("seaborn-v0_8-whitegrid")
 
# Chart A — Ticket volume by type
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(type_volume["Ticket Type"], type_volume["Ticket Count"], color="#2E86AB")
ax.bar(top3["Ticket Type"], top3["Ticket Count"], color="#A23B72")
ax.set_title(f"Ticket Volume by Type\nTop 3 drive {top3_share:.0f}% of total volume", fontsize=12)
ax.set_ylabel("Ticket Count")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_volume_by_type.png", dpi=150)
plt.close()
 
# Chart B — Resolution time by type
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(res_time_by_type["Ticket Type"], res_time_by_type["Avg_Hours"], color="#F18F01")
ax.set_title("Avg Resolution Time by Ticket Type (valid durations only)", fontsize=12)
ax.set_ylabel("Avg Hours to Resolve")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_resolution_time_by_type.png", dpi=150)
plt.close()
 
# Chart C — Repeat vs one-off: volume share & satisfaction
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].pie(
    [n_repeat_customers, n_unique_customers - n_repeat_customers],
    labels=["Repeat", "One-off"], autopct="%1.1f%%",
    colors=["#A23B72", "#C9C9C9"]
)
axes[0].set_title("Customers: Repeat vs One-off")
 
axes[1].bar(
    ["Repeat\nComplainants", "One-off\nComplainants"],
    [avg_satisfaction_repeat, avg_satisfaction_oneoff],
    color=["#A23B72", "#C9C9C9"]
)
axes[1].set_title("Avg Satisfaction Rating")
axes[1].set_ylim(0, 5)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_repeat_vs_oneoff.png", dpi=150)
plt.close()
 
print("Exported 3 charts to /outputs")
print("\nDONE.")

STEP 1: RAW DATA OVERVIEW
Rows: 8,469 | Columns: 17

Nulls per column:
Resolution                      5700
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64

Ticket Status breakdown:
Ticket Status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64

STEP 2: CLEANING
Removed 0 duplicate Ticket IDs
Closed tickets: 2,769
Closed tickets with invalid (<=0 hr) duration: 1,367 (49.4%) -- flagged, excluded from duration averages

Unique customers: 8,320
Repeat complainants (>1 ticket): 139 (1.7% of unique customers)

Final cleaned shape: (8469, 22)

STEP 3: ANALYSIS

Ticket volume by type:
         Ticket Type  Ticket Count  % of Total
      Refund request          1752        20.7
     Technical issue          1747        20.6
Cancellation request          1695        20.0
     Product inquiry          1641        19.4
     Billing inquiry     